# Colab Generation Improvement Experiments

이 노트북은 이미 생성된 baseline 결과를 건드리지 않고, generation 품질 개선 실험을 작은 subset부터 반복하기 위한 실험용 노트북입니다.

실험 축:
1. generation context 구성 개선
2. 질문 유형별 context 주입 방식 개선
3. 예산/날짜/제출서류 등 필드별 후보값 추출 강화
4. 후처리 guardrail 강화
5. 사람이 평가 기준 정리
6. 그 기준을 기반으로 Judge LLM 자동 평가 설계
7. 그 이후 모델 변경 또는 튜닝

source_store는 검색 대상이 아니라, 검색된 chunk의 `source_ref.source_store_id`로 더 긴 원문을 찾아 generation context에 보강하는 lookup 저장소로 사용합니다.

In [10]:
# 1. Experiment config
from pathlib import Path

REPO_URL = 'https://github.com/beomsookim1020/chatbot.git'
REPO_BRANCH = 'colab-generation'
PROJECT_DIR = Path('/content/chatbot')

DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_inputs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_outputs')
DRIVE_EXPERIMENT_ROOT = DRIVE_OUTPUT_ROOT / 'generation_improvement_experiments'

PREDICTION_REL = Path(
    'outputs/predictions/'
    '74_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse250_'
    'soyeon_125_kure_chroma_hnsw_tuned_canonical.jsonl'
)
CHUNK_SIDECAR_REL = Path('indexes/chroma_kure_v1_soyeon_125_260520_chunks_v2_125/chunks.jsonl')

# source_store는 아래 Drive 경로를 사용합니다.
# MyDrive/chatbot_colab_inputs/data/source_store_v2_690.jsonl
SOURCE_STORE_FILENAME = 'source_store_v2_690.jsonl'
SOURCE_STORE_REL = Path('data') / SOURCE_STORE_FILENAME

# 실험용 eval 파일: baseline에서 틀린 대표 30개 문제만 담은 batch-format CSV입니다.
EXPERIMENT_EVAL_FILENAME = 'representative_wrong_30_eval_batch_format.csv'

# 0이면 실험용 eval 파일 전체를 사용합니다.
EXPERIMENT_LIMIT = 0
EXPERIMENT_IDS = []  # 특정 id만 보고 싶으면 ['Q001', 'Q002']처럼 입력

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_NEW_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.9

# Judge는 설계 셀까지 만들고 기본 실행은 꺼둡니다.
RUN_JUDGE = False
JUDGE_MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'

print('PROJECT_DIR:', PROJECT_DIR)
print('DRIVE_INPUT_ROOT:', DRIVE_INPUT_ROOT)
print('DRIVE_EXPERIMENT_ROOT:', DRIVE_EXPERIMENT_ROOT)
print('MODEL_NAME:', MODEL_NAME)
print('EXPERIMENT_LIMIT:', EXPERIMENT_LIMIT)

PROJECT_DIR: /content/chatbot
DRIVE_INPUT_ROOT: /content/drive/MyDrive/chatbot_colab_inputs
DRIVE_EXPERIMENT_ROOT: /content/drive/MyDrive/chatbot_colab_outputs/generation_improvement_experiments
MODEL_NAME: Qwen/Qwen2.5-3B-Instruct
EXPERIMENT_LIMIT: 0


In [11]:
# 2. Runtime setup: GPU, Drive, repo, packages
import shutil
import subprocess
import sys

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Colab GPU runtime이 아닙니다. Runtime > Change runtime type > GPU로 바꿔주세요.')
subprocess.run(['nvidia-smi'], check=True)

from google.colab import drive
drive.mount('/content/drive')

def run_cmd(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)))
    subprocess.run([str(part) for part in cmd], cwd=str(cwd) if cwd else None, check=True)

if (PROJECT_DIR / '.git').exists():
    run_cmd(['git', 'fetch', 'origin', REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(['git', 'checkout', REPO_BRANCH], cwd=PROJECT_DIR)
    run_cmd(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=PROJECT_DIR)
else:
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_cmd(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, PROJECT_DIR])

packages = [
    '-r', str(PROJECT_DIR / 'requirements.txt'),
    'transformers>=4.45.0',
    'accelerate',
    'sentencepiece',
    'safetensors',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
# Colab에서 requirements 설치 후 pandas/numpy ABI가 꼬이는 경우를 줄이기 위해 버전을 고정합니다.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir',
    'numpy==1.26.4',
    'pandas==2.2.2',
], check=True)
print('Runtime ready.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin colab-generation
$ git checkout colab-generation
$ git pull --ff-only origin colab-generation
Runtime ready.


In [12]:
# 3. Copy inputs from Drive to Colab local project
import shutil

DRIVE_EVAL_DIR = DRIVE_INPUT_ROOT / 'data/eval'
DRIVE_EXPERIMENT_EVAL_FILE = DRIVE_EVAL_DIR / EXPERIMENT_EVAL_FILENAME
DRIVE_PREDICTIONS = DRIVE_INPUT_ROOT / PREDICTION_REL

def first_existing(paths):
    return next((path for path in paths if path and path.exists()), None)

chunk_candidates = [DRIVE_INPUT_ROOT / CHUNK_SIDECAR_REL]
chunk_candidates.append((DRIVE_INPUT_ROOT / CHUNK_SIDECAR_REL).with_suffix('.json'))
index_root = DRIVE_INPUT_ROOT / 'indexes'
if index_root.exists():
    chunk_candidates.extend(sorted(index_root.glob('**/chunks.jsonl')))
    chunk_candidates.extend(sorted(index_root.glob('**/chunks.json')))
DRIVE_CHUNK_SIDECAR = first_existing(chunk_candidates)

DRIVE_SOURCE_STORE = DRIVE_INPUT_ROOT / SOURCE_STORE_REL

missing = []
if not DRIVE_EXPERIMENT_EVAL_FILE.exists():
    missing.append(str(DRIVE_EXPERIMENT_EVAL_FILE))
if not DRIVE_PREDICTIONS.exists():
    missing.append(str(DRIVE_PREDICTIONS))
if missing:
    raise FileNotFoundError('Missing required inputs:\n' + '\n'.join(missing[:20]))

LOCAL_EVAL_DIR = PROJECT_DIR / 'data/experiment_eval'
LOCAL_PREDICTIONS = PROJECT_DIR / PREDICTION_REL
LOCAL_CHUNK_SIDECAR = PROJECT_DIR / CHUNK_SIDECAR_REL if DRIVE_CHUNK_SIDECAR else None
LOCAL_SOURCE_STORE = PROJECT_DIR / 'data/source_store/source_store.jsonl' if DRIVE_SOURCE_STORE else None

if LOCAL_EVAL_DIR.exists():
    shutil.rmtree(LOCAL_EVAL_DIR)
LOCAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_EXPERIMENT_EVAL_FILE, LOCAL_EVAL_DIR / EXPERIMENT_EVAL_FILENAME)

LOCAL_PREDICTIONS.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(DRIVE_PREDICTIONS, LOCAL_PREDICTIONS)

if DRIVE_CHUNK_SIDECAR:
    LOCAL_CHUNK_SIDECAR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_CHUNK_SIDECAR, LOCAL_CHUNK_SIDECAR)
if DRIVE_SOURCE_STORE:
    LOCAL_SOURCE_STORE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_SOURCE_STORE, LOCAL_SOURCE_STORE)

print('experiment eval:', LOCAL_EVAL_DIR / EXPERIMENT_EVAL_FILENAME)
print('predictions:', LOCAL_PREDICTIONS)
print('chunk sidecar:', LOCAL_CHUNK_SIDECAR)
print('source_store:', LOCAL_SOURCE_STORE)
if not LOCAL_SOURCE_STORE:
    print('WARN: source_store를 찾지 못했습니다. source_store variant는 skip 또는 source_store off로 동작합니다.')

experiment eval: /content/chatbot/data/experiment_eval/representative_wrong_30_eval_batch_format.csv
predictions: /content/chatbot/outputs/predictions/74_dense_conditional_qdecomp_v2_rrf_multi_relaxed_filter_kept_per50_diverse250_soyeon_125_kure_chroma_hnsw_tuned_canonical.jsonl
chunk sidecar: /content/chatbot/indexes/chroma_kure_v1_soyeon_125_260520_chunks_v2_125/chunks.jsonl
source_store: /content/chatbot/data/source_store/source_store.jsonl


In [13]:
# 4. Load eval/prediction rows and source_store
import json
import os
import re
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = str(PROJECT_DIR)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
EVAL_SRC = PROJECT_DIR / 'eval/evaluation/src'
if str(EVAL_SRC) not in sys.path:
    sys.path.insert(0, str(EVAL_SRC))

from rag_eval.loaders import load_eval_csvs, load_predictions_jsonl, merge_eval_predictions
from src.generation import classify_question, dedupe_repeated_lines
from src.generator import HuggingFaceGenerator

def read_json_or_jsonl(path: Path) -> list[dict[str, Any]]:
    text = path.read_text(encoding='utf-8')
    if not text.strip():
        return []
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ['chunks', 'records', 'items', 'data']:
            if isinstance(data.get(key), list):
                return data[key]
    raise ValueError(f'Unsupported JSON/JSONL format: {path}')

def first_non_empty(row, *keys):
    for key in keys:
        value = row.get(key)
        if value not in (None, ''):
            return value
    return None

eval_df = load_eval_csvs(LOCAL_EVAL_DIR, canonical_only=False)
pred_df = load_predictions_jsonl(LOCAL_PREDICTIONS)
merged = merge_eval_predictions(eval_df, pred_df)
merged = merged[~merged['prediction_missing']].copy()

if EXPERIMENT_IDS:
    merged = merged[merged['id'].isin(EXPERIMENT_IDS)]
elif EXPERIMENT_LIMIT and EXPERIMENT_LIMIT > 0:
    merged = merged.head(EXPERIMENT_LIMIT)

rows = []
for _, row in merged.iterrows():
    retrieved_contexts = row.get('retrieved_contexts')
    rows.append({
        'id': row.get('id'),
        'type': row.get('type'),
        'difficulty': row.get('difficulty'),
        'question': first_non_empty(row, 'question_eval', 'question', 'question_pred'),
        'ground_truth_answer': row.get('ground_truth_answer'),
        'ground_truth_docs': row.get('ground_truth_doc_list'),
        'metadata_filter': row.get('metadata_filter_obj'),
        'retrieved_contexts': retrieved_contexts if isinstance(retrieved_contexts, list) else [],
    })

source_store_by_id = {}
source_store_by_evidence_id = {}
if LOCAL_SOURCE_STORE and LOCAL_SOURCE_STORE.exists():
    for item in read_json_or_jsonl(LOCAL_SOURCE_STORE):
        sid = item.get('source_store_id') or item.get('id') or item.get('source_id')
        eid = item.get('evidence_id') or item.get('evidence')
        if sid:
            source_store_by_id[str(sid)] = item
        if eid:
            source_store_by_evidence_id[str(eid)] = item

print('selected rows:', len(rows))
print('source_store rows:', len(source_store_by_id))

selected rows: 30
source_store rows: 101737


In [14]:
# 5. Context, source_store, field candidates, guardrails
MONEY_PATTERN = re.compile(r'(?:금\s*)?(?:[0-9]{1,3}(?:,[0-9]{3})+|[0-9]+(?:\.[0-9]+)?)\s*(?:원|천원|만원|억원|백만원|KRW|부가세|VAT)|KRW\s*:?\s*[0-9,]+')
DATE_PATTERN = re.compile(r'(?:20\d{2}|\d{2})[.\-/년]\s*\d{1,2}[.\-/월]\s*\d{1,2}(?:일)?|\d{1,2}\s*월\s*\d{1,2}\s*일|\d{1,2}:\d{2}')
DOC_PATTERN = re.compile(r'[^\s:;]+\.(?:hwp|hwpx|pdf|docx?|xlsx?)', re.IGNORECASE)
SUBMISSION_WORDS = ('제출서류', '제출 서류', '제안서', '입찰참가', '구비서류', '첨부서류', '서식')
QUALIFICATION_WORDS = ('자격', '참가자격', '입찰자격', '실적', '제한', '공동수급')

def coerce_text(value):
    return '' if value is None else str(value)

def unique(values, limit=30):
    result = []
    for value in values:
        text = coerce_text(value).strip()
        if text and text not in result:
            result.append(text)
        if len(result) >= limit:
            break
    return result

def parse_ref(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str) and value.strip():
        try:
            data = json.loads(value)
            return data if isinstance(data, dict) else {}
        except json.JSONDecodeError:
            return {}
    return {}

def context_text(context):
    return coerce_text(context.get('text') or context.get('content') or context.get('page_content'))

def source_item_text(item):
    parts = []
    for key in ['text', 'content', 'raw_text', 'source_text', 'table_text', 'markdown', 'html']:
        value = item.get(key)
        if value:
            parts.append(coerce_text(value))
    return '\n'.join(parts)

def source_ref_from_context(context):
    metadata = context.get('metadata') or {}
    ref = parse_ref(context.get('source_ref') or metadata.get('source_ref'))
    sid = ref.get('source_store_id') or context.get('source_store_id') or metadata.get('source_store_id')
    eid = ref.get('evidence_id') or context.get('evidence_id') or metadata.get('evidence_id')
    return {'source_store_id': sid, 'evidence_id': eid}

def normalize_contexts(retrieved_contexts, max_chars):
    records = []
    for rank, context in enumerate(retrieved_contexts, 1):
        metadata = context.get('metadata') or {}
        text = context_text(context)
        records.append({
            'rank': rank,
            'filename': context.get('filename') or metadata.get('source_file'),
            'doc_id': context.get('doc_id') or metadata.get('doc_id'),
            'chunk_id': context.get('chunk_id') or metadata.get('chunk_id'),
            'score': context.get('score'),
            'metadata': metadata,
            'source_ref': source_ref_from_context(context),
            'text': text[:max_chars] if max_chars else text,
            'full_text': text,
        })
    return records

def lookup_source_records(context_records, variant):
    if not variant.get('use_source_store') or not source_store_by_id:
        return []
    max_items = variant.get('max_source_items', 5)
    max_chars = variant.get('source_max_chars', 1600)
    seen = set()
    records = []
    for context in context_records:
        ref = context.get('source_ref') or {}
        item = None
        sid = ref.get('source_store_id')
        eid = ref.get('evidence_id')
        if sid:
            item = source_store_by_id.get(str(sid))
        if item is None and eid:
            item = source_store_by_evidence_id.get(str(eid))
        if not item:
            continue
        key = str(item.get('source_store_id') or item.get('id') or sid or eid)
        if key in seen:
            continue
        seen.add(key)
        records.append({
            'source_store_id': key,
            'evidence_id': item.get('evidence_id') or eid,
            'filename': item.get('filename') or item.get('source_file') or context.get('filename'),
            'text': source_item_text(item)[:max_chars],
            'metadata': item.get('metadata') or {},
            'from_rank': context.get('rank'),
        })
        if len(records) >= max_items:
            break
    return records

def score_for_question_type(text, question_type):
    if question_type == 'budget':
        keys = ('예산', '금액', '기초금액', '추정가격', '사업비', '원')
    elif question_type == 'date_or_period':
        keys = ('기간', '마감', '제출', '접수', '계약기간', '공고일')
    elif question_type == 'submission_documents':
        keys = SUBMISSION_WORDS
    elif question_type == 'qualification':
        keys = QUALIFICATION_WORDS
    else:
        keys = ()
    return sum(1 for key in keys if key in text)

def extract_field_candidates(context_records, source_records, enhanced=False):
    texts = [record.get('text') or '' for record in context_records] + [record.get('text') or '' for record in source_records]
    metadata_values = []
    for record in context_records:
        metadata = record.get('metadata') or {}
        metadata_values.extend([metadata.get('issuer'), metadata.get('project_name'), metadata.get('budget'), metadata.get('section_path')])
        for key in ['amounts', 'dates', 'submission_documents', 'qualifications']:
            value = metadata.get(key)
            if isinstance(value, list):
                metadata_values.extend(value)
            elif value:
                metadata_values.append(value)
    all_text = '\n'.join(texts)
    candidates = {
        'amounts': unique(MONEY_PATTERN.findall(all_text), 40),
        'dates': unique(DATE_PATTERN.findall(all_text), 40),
        'docs': unique(DOC_PATTERN.findall(all_text), 40),
        'metadata_values': unique(metadata_values, 40),
    }
    if enhanced:
        lines = [re.sub(r'\s+', ' ', line).strip() for line in all_text.splitlines()]
        candidates['submission_lines'] = unique([line for line in lines if any(word in line for word in SUBMISSION_WORDS)], 20)
        candidates['qualification_lines'] = unique([line for line in lines if any(word in line for word in QUALIFICATION_WORDS)], 20)
        candidates['budget_lines'] = unique([line for line in lines if MONEY_PATTERN.search(line) and any(word in line for word in ('예산', '금액', '가격', '사업비'))], 20)
        candidates['date_lines'] = unique([line for line in lines if DATE_PATTERN.search(line)], 20)
    return {key: value for key, value in candidates.items() if value}

def render_prompt(row, variant):
    if variant.get('context_builder_v2'):
        return render_prompt_context_builder_v2(row, variant)
    question = row['question']
    question_type = classify_question(question)
    context_records = normalize_contexts(row['retrieved_contexts'], variant.get('context_max_chars', 1000))
    source_records = lookup_source_records(context_records, variant)
    if variant.get('typed_context'):
        source_records = sorted(source_records, key=lambda record: score_for_question_type(record.get('text') or '', question_type), reverse=True)
        context_records = sorted(context_records, key=lambda record: score_for_question_type(record.get('text') or '', question_type), reverse=True)
    fields = extract_field_candidates(context_records, source_records, enhanced=variant.get('enhanced_fields', False))
    type_rule = {
        'budget': '금액 후보를 먼저 확인하고 기초금액/추정가격/사업예산/사업비를 섞지 않는다.',
        'date_or_period': '날짜 후보를 먼저 확인하고 제출마감/접수기간/계약기간을 구분한다.',
        'submission_documents': '제출서류와 첨부/서식 목록은 누락 없이 정리하되 context에 없는 서류는 만들지 않는다.',
        'qualification': '참가자격/제한요건/실적요건을 구분한다.',
    }.get(question_type, '질문에 직접 답하는 근거 문장을 우선한다.')
    parts = [
        '너는 RFP 문서 기반 QA assistant다.',
        '',
        '규칙:',
        '1. 제공된 Context 안의 정보만 사용한다.',
        '2. Context에 없으면 "문서에서 확인할 수 없습니다"라고 답한다.',
        '3. 금액, 날짜, 기간, 공고번호는 원문 표현을 보존한다.',
        '4. 답변에는 근거 문서명과 근거 문장을 함께 제시한다.',
        '5. 후보가 여러 개면 하나로 단정하지 말고 후보를 나눠 말한다.',
        f'질문 유형: {question_type}',
        f'질문 유형별 규칙: {type_rule}',
        '',
        '[Question]',
        question,
    ]
    if variant.get('enhanced_fields') and fields:
        parts.extend(['', '[필드별 후보값]'])
        for key, values in fields.items():
            parts.append(f'- {key}: ' + '; '.join(values[:12]))
    parts.extend(['', '[검색 chunk context]'])
    for record in context_records:
        parts.append(f"--- rank {record.get('rank')} | {record.get('filename')} | chunk {record.get('chunk_id')} ---")
        parts.append(record.get('text') or '')
    if source_records:
        parts.extend(['', '[source_store 확장 원문]'])
        for record in source_records:
            parts.append(f"--- source_store {record.get('source_store_id')} | from rank {record.get('from_rank')} | {record.get('filename')} ---")
            parts.append(record.get('text') or '')
    parts.extend(['', '답변 형식:', '[답변]', '- ...', '', '[근거]', '- 문서명: ...', '- 근거 문장: "..."', '', '[주의/불확실성]', '- ...'])
    return {
        'prompt': '\n'.join(parts),
        'question_type': question_type,
        'context_records': context_records,
        'source_records': source_records,
        'field_candidates': fields,
    }


# 공유 문서 v2에서 가져온 Context Builder 핵심 정책입니다.
FINAL_BUDGET_FACT_TYPES = {'project_budget', 'budget', 'estimated_price', 'base_amount'}
BUDGET_BLOCKED_FACT_TYPES = {'threshold_budget', 'payment_terms', 'deposit', 'bid_deposit', 'contract_deposit', 'performance_bond', 'penalty', 'warranty'}

INTENT_REQUIRED_FACT_TYPES = {
    'budget_lookup': ['project_budget', 'budget', 'estimated_price', 'base_amount'],
    'budget_difference': ['project_budget', 'budget', 'estimated_price', 'base_amount'],
    'budget_sum': ['project_budget', 'budget', 'estimated_price', 'base_amount'],
    'budget_ratio': ['project_budget', 'budget', 'estimated_price', 'base_amount'],
    'submission_documents': ['submission_documents'],
    'submission_logistics': ['submission_logistics'],
    'project_duration': ['project_duration', 'contract_period', 'duration'],
    'submission_deadline': ['submission_deadline', 'submission_logistics'],
    'eligibility': ['eligibility', 'qualification', 'requirements'],
    'purpose_summary': ['document_summary', 'business_type', 'requirements'],
}

INTENT_PREFERRED_CHUNK_TYPES = {
    'budget_lookup': ['fact_candidates', 'text', 'table'],
    'budget_difference': ['fact_candidates', 'text', 'table'],
    'budget_sum': ['fact_candidates', 'text', 'table'],
    'budget_ratio': ['fact_candidates', 'text', 'table'],
    'submission_documents': ['fact_candidates', 'table', 'text'],
    'submission_logistics': ['fact_candidates', 'text', 'table'],
    'project_duration': ['fact_candidates', 'text'],
    'submission_deadline': ['fact_candidates', 'text'],
    'eligibility': ['fact_candidates', 'text', 'table'],
    'purpose_summary': ['text', 'fact_candidates', 'table'],
}

def normalize_for_match(value):
    return re.sub(r'\s+', '', coerce_text(value).lower())

def record_value(record, key, default=''):
    metadata = record.get('metadata') or {}
    return record.get(key) or metadata.get(key) or default

def bool_record_value(record, key):
    value = record_value(record, key, False)
    if isinstance(value, str):
        return value.strip().lower() in {'1', 'true', 'yes', 'y', 'allow'}
    return bool(value)

def analyze_question_v2(question):
    q = coerce_text(question)
    compact = normalize_for_match(q)
    base_type = classify_question(q)
    question_types = [base_type]
    intents = []
    answer_type = base_type

    if base_type == 'budget':
        if any(word in q for word in ['차액', '차이', '얼마 차', '비교']):
            intents.append('budget_difference')
        elif any(word in q for word in ['합계', '합산', '총액', '총 예산']):
            intents.append('budget_sum')
        elif any(word in q for word in ['%', '퍼센트', '비율', '나누', '단가', '계산', '산술']):
            intents.append('budget_ratio')
        else:
            intents.append('budget_lookup')
        answer_type = 'budget'
    elif base_type == 'date_or_period':
        if any(word in q for word in ['마감', '제출', '접수', '입찰', '개찰']):
            intents.append('submission_deadline')
        if any(word in q for word in ['계약기간', '사업기간', '수행기간', '용역기간', '기간']):
            intents.append('project_duration')
        if not intents:
            intents.extend(['project_duration', 'submission_deadline'])
        answer_type = 'date_or_period'
    elif base_type == 'submission_documents':
        intents.append('submission_documents')
        if any(word in q for word in ['방법', '장소', '마감', '시간', '접수']):
            intents.append('submission_logistics')
        answer_type = 'submission_documents'
    elif base_type == 'qualification':
        intents.append('eligibility')
        answer_type = 'qualification'
    else:
        if any(word in q for word in ['목적', '배경', '요약', '무엇을', '내용']):
            intents.append('purpose_summary')
        else:
            intents.append('general')
        answer_type = 'general'

    is_multi_doc = any(word in q for word in ['각각', '비교', '두 문서', '여러', '모든 문서'])
    intent_plan = []
    for intent in intents:
        intent_plan.append({
            'intent': intent,
            'required_fact_types': INTENT_REQUIRED_FACT_TYPES.get(intent, []),
            'preferred_chunk_types': INTENT_PREFERRED_CHUNK_TYPES.get(intent, ['text', 'table', 'fact_candidates']),
            'requires_computation': intent in {'budget_difference', 'budget_sum', 'budget_ratio'},
        })
    return {
        'question_types': question_types,
        'question_type': base_type,
        'intent_slots': intents,
        'intent_plan': intent_plan,
        'answer_type': answer_type,
        'is_multi_doc': is_multi_doc,
        'needs_synthesis': is_multi_doc or len(intents) > 1 or any(intent in {'purpose_summary'} for intent in intents),
    }

def score_evidence_v2(record, analysis):
    text = record.get('text') or ''
    fact_type = coerce_text(record_value(record, 'fact_type'))
    chunk_type = coerce_text(record_value(record, 'chunk_type'))
    answer_policy = coerce_text(record_value(record, 'answer_policy'))
    question_types = set(analysis.get('question_types') or [])
    intents = analysis.get('intent_slots') or []
    score = 0.0

    rank = record.get('rank') or record.get('from_rank') or 99
    try:
        score += max(0, 40 - float(rank) * 3)
    except Exception:
        score += 5
    if record.get('is_source_store'):
        score += 8

    required = set()
    preferred_chunks = []
    for intent in intents:
        required.update(INTENT_REQUIRED_FACT_TYPES.get(intent, []))
        preferred_chunks.extend(INTENT_PREFERRED_CHUNK_TYPES.get(intent, []))
    if fact_type in required:
        score += 38
    if chunk_type in preferred_chunks:
        score += max(6, 18 - preferred_chunks.index(chunk_type) * 4)

    if fact_type == 'document_identity' or answer_policy == 'route_only_not_final_answer':
        score += 8
        if not analysis.get('is_multi_doc') and not analysis.get('needs_synthesis'):
            score -= 45

    if 'budget' in question_types:
        budget_enabled = bool_record_value(record, 'budget_answer_enabled')
        if budget_enabled:
            score += 35
        if fact_type in FINAL_BUDGET_FACT_TYPES:
            score += 25
        if fact_type in BUDGET_BLOCKED_FACT_TYPES:
            score -= 90
        if chunk_type == 'fact_candidates' and not budget_enabled and fact_type:
            score -= 35
        if MONEY_PATTERN.search(text):
            score += 8

    if 'submission_documents' in question_types:
        if fact_type == 'submission_documents':
            score += 35
        if any(word in text for word in SUBMISSION_WORDS):
            score += 12
    if 'qualification' in question_types:
        if fact_type in {'eligibility', 'qualification', 'requirements'} or bool_record_value(record, 'eligibility_answer_enabled'):
            score += 30
        if any(word in text for word in QUALIFICATION_WORDS):
            score += 10
    if 'date_or_period' in question_types and DATE_PATTERN.search(text):
        score += 15
    return score

def build_evidence_blocks_v2(context_records, source_records, analysis, variant):
    blocks = []
    for record in context_records:
        block = dict(record)
        block['is_source_store'] = False
        block['evidence_score'] = score_evidence_v2(block, analysis)
        blocks.append(block)
    for record in source_records:
        block = dict(record)
        block['rank'] = None
        block['chunk_id'] = record.get('evidence_id')
        block['source_ref'] = {'source_store_id': record.get('source_store_id'), 'evidence_id': record.get('evidence_id')}
        block['is_source_store'] = True
        block['evidence_score'] = score_evidence_v2(block, analysis)
        blocks.append(block)
    blocks.sort(key=lambda item: item.get('evidence_score', 0), reverse=True)
    return blocks[:variant.get('max_evidence_blocks', 8)]

def format_context_builder_v2(question, analysis, fields, evidence_blocks, max_chars):
    lines = [
        '[핵심 추출값 요약]',
        f"질문유형: {', '.join(analysis.get('question_types', []))}",
        f"답변유형: {analysis.get('answer_type', 'unknown')}",
        f"의도 슬롯: {', '.join(analysis.get('intent_slots', []))}",
    ]
    if analysis.get('intent_plan'):
        lines.extend(['', '[intent plan - 질문 안의 하위 요청]'])
        for plan in analysis['intent_plan']:
            lines.append(
                f"- intent={plan.get('intent')} | required_fact_types={', '.join(plan.get('required_fact_types') or []) or '-'} | "
                f"preferred_chunk_types={', '.join(plan.get('preferred_chunk_types') or []) or '-'} | "
                f"requires_computation={plan.get('requires_computation', False)}"
            )
    if fields:
        lines.extend(['', '[필드별 후보값 - 원문에서 추출]'])
        for key, values in fields.items():
            lines.append(f'- {key}: ' + '; '.join(values[:10]))
    lines.extend(['', '[선별 근거 blocks]'])
    for idx, block in enumerate(evidence_blocks, 1):
        metadata = block.get('metadata') or {}
        fact_type = record_value(block, 'fact_type', '-')
        chunk_type = record_value(block, 'chunk_type', '-')
        answer_policy = record_value(block, 'answer_policy', '-')
        budget_enabled = record_value(block, 'budget_answer_enabled', '-')
        source_kind = 'source_store' if block.get('is_source_store') else 'retrieved_chunk'
        lines.append(
            f"근거 {idx}: kind={source_kind} | score={round(block.get('evidence_score', 0), 2)} | "
            f"source_file={block.get('filename') or '-'} | chunk_id={block.get('chunk_id') or '-'} | "
            f"chunk_type={chunk_type} | fact_type={fact_type} | answer_policy={answer_policy} | "
            f"budget_answer_enabled={budget_enabled}"
        )
        lines.append((block.get('text') or '')[:1200])
    context_text = '\n'.join(lines)
    return context_text[:max_chars]

def render_prompt_context_builder_v2(row, variant):
    question = row['question']
    analysis = analyze_question_v2(question)
    context_records = normalize_contexts(row['retrieved_contexts'], variant.get('context_max_chars', 1000))
    source_records = lookup_source_records(context_records, variant)
    fields = extract_field_candidates(context_records, source_records, enhanced=True)
    evidence_blocks = build_evidence_blocks_v2(context_records, source_records, analysis, variant)
    context_text = format_context_builder_v2(
        question,
        analysis,
        fields,
        evidence_blocks,
        max_chars=variant.get('max_context_chars_v2', 9000),
    )
    prompt = f"""너는 RFP 문서 기반 QA assistant다.

[질문]
{question}

[Context]
{context_text}

[답변 규칙]
- 제공된 Context 안의 정보만 사용한다.
- fact_type=document_identity 또는 answer_policy=route_only_not_final_answer 근거는 문서 식별 신호로만 사용하고, 숫자/날짜/금액의 최종 근거로 사용하지 않는다.
- 예산 질문에서 threshold_budget, payment_terms, deposit 계열 값은 입찰자격/지급조건 신호일 뿐 사업예산의 최종값으로 쓰지 않는다.
- 예산 답변은 fact_type이 project_budget, budget, estimated_price, base_amount 이거나 budget_answer_enabled=true인 근거를 우선한다.
- target이 있는 질문에서는 source_file이 맞는 문서의 값만 사용한다. 같은 기관의 다른 사업 예산을 대체값으로 쓰지 않는다.
- 답변의 숫자, 날짜, 서류명은 [선별 근거 blocks] 또는 [필드별 후보값]에 있는 문자열만 사용한다.
- 근거가 부족하면 추측하지 말고 "문서에서 확인할 수 없습니다"라고 답한다.

답변 형식:
[답변]
- ...

[근거]
- 문서명: ...
- 근거 문장: "..."

[주의/불확실성]
- ...""".strip()
    return {
        'prompt': prompt,
        'question_type': analysis.get('question_type'),
        'context_records': context_records,
        'source_records': source_records,
        'field_candidates': fields,
        'question_analysis': analysis,
        'evidence_blocks': evidence_blocks,
    }

def guardrail(answer, prompt_bundle, variant):
    context_text = '\n'.join([record.get('text') or '' for record in prompt_bundle['context_records'] + prompt_bundle['source_records']])
    numeric_tokens = unique(MONEY_PATTERN.findall(answer) + DATE_PATTERN.findall(answer), 50)
    unsupported_numbers = [token for token in numeric_tokens if token not in context_text]
    docs = unique([record.get('filename') for record in prompt_bundle['context_records'] + prompt_bundle['source_records'] if record.get('filename')], 50)
    mentioned_docs = unique(DOC_PATTERN.findall(answer), 50)
    warnings = []
    if unsupported_numbers:
        warnings.append('answer_has_number_or_date_not_in_context')
    if docs and not any(doc and doc in answer for doc in docs) and not mentioned_docs:
        warnings.append('missing_source_citation')
    if '문서에서 확인할 수 없습니다' in answer and any(prompt_bundle.get('field_candidates', {}).values()):
        warnings.append('unknown_answer_despite_field_candidates')
    if variant.get('strict_guardrail') and unsupported_numbers:
        warnings.append('strict_guardrail_review_required')
    return {
        'warnings': warnings,
        'unsupported_numbers_or_dates': unsupported_numbers,
        'mentioned_docs': mentioned_docs,
        'confidence': 'low' if unsupported_numbers else ('medium' if warnings else 'high'),
    }


In [15]:
# 6. Experiment variants: 1~4번 개선 축
EXPERIMENT_VARIANTS = {
    'A_baseline_like_context1000': {
        'description': '현재 baseline과 비슷한 context 구성. source_store off.',
        'context_max_chars': 1000,
        'use_source_store': False,
        'enhanced_fields': False,
        'typed_context': False,
        'strict_guardrail': False,
    },
    'B_long_context1500': {
        'description': 'generation context 길이만 늘린 실험.',
        'context_max_chars': 1500,
        'use_source_store': False,
        'enhanced_fields': False,
        'typed_context': False,
        'strict_guardrail': False,
    },
    'C_source_store_basic': {
        'description': '검색 chunk의 source_store_id로 긴 원문을 추가 주입.',
        'context_max_chars': 1000,
        'use_source_store': True,
        'max_source_items': 5,
        'source_max_chars': 1600,
        'enhanced_fields': False,
        'typed_context': False,
        'strict_guardrail': False,
    },
    'D_typed_source_store': {
        'description': '질문 유형별로 관련성이 높은 검색 chunk/source_store 원문을 앞쪽에 배치.',
        'context_max_chars': 1000,
        'use_source_store': True,
        'max_source_items': 6,
        'source_max_chars': 1800,
        'enhanced_fields': False,
        'typed_context': True,
        'strict_guardrail': False,
    },
    'E_fields_v2': {
        'description': '예산/날짜/제출서류/자격 후보값을 강화해서 prompt 앞쪽에 주입.',
        'context_max_chars': 1000,
        'use_source_store': True,
        'max_source_items': 6,
        'source_max_chars': 1800,
        'enhanced_fields': True,
        'typed_context': True,
        'strict_guardrail': False,
    },
    'G_context_builder_v2': {
        'description': '공유 문서 v2 반영: intent plan, fact_type/answer_policy 점수화, source_store lookup, strict grounding prompt.',
        'context_max_chars': 1000,
        'use_source_store': True,
        'max_source_items': 6,
        'source_max_chars': 1800,
        'max_evidence_blocks': 8,
        'max_context_chars_v2': 9000,
        'enhanced_fields': True,
        'typed_context': True,
        'strict_guardrail': True,
        'context_builder_v2': True,
    },
    'F_guardrail_v2': {
        'description': 'fields_v2에 후처리 guardrail warning을 강화한 실험.',
        'context_max_chars': 1000,
        'use_source_store': True,
        'max_source_items': 6,
        'source_max_chars': 1800,
        'enhanced_fields': True,
        'typed_context': True,
        'strict_guardrail': True,
    },
}

# 현재는 기존 최고 후보였던 E와 공유 문서 v2를 반영한 G를 비교합니다.
VARIANTS_TO_RUN = ['E_fields_v2', 'G_context_builder_v2']
for name in VARIANTS_TO_RUN:
    print(name, '-', EXPERIMENT_VARIANTS[name]['description'])


E_fields_v2 - 예산/날짜/제출서류/자격 후보값을 강화해서 prompt 앞쪽에 주입.
G_context_builder_v2 - 공유 문서 v2 반영: intent plan, fact_type/answer_policy 점수화, source_store lookup, strict grounding prompt.


## 적용한 Context Builder v2 요약

공유 HTML 문서의 전체 원본 코드를 그대로 옮긴 것은 아니고, 현재 실험 노트북에 바로 비교 가능한 `G_context_builder_v2` variant로 핵심 아이디어를 적용했습니다.

- 질문을 예산/기간/제출서류/자격 같은 유형과 `budget_lookup`, `budget_ratio`, `submission_documents` 같은 하위 의도로 나눕니다.
- `fact_type`, `chunk_type`, `answer_policy`, `budget_answer_enabled`를 읽어서 근거 block에 점수를 매깁니다.
- `document_identity`나 `route_only_not_final_answer`는 문서 식별에는 쓰되 최종 금액/날짜 근거로 쓰지 못하게 낮은 점수를 줍니다.
- 예산 질문에서는 `project_budget`, `estimated_price`, `base_amount`처럼 최종 답변에 쓸 수 있는 fact를 우선하고, `threshold_budget`, `payment_terms`, `deposit` 계열은 차단합니다.
- 검색 chunk의 `source_store_id`로 긴 원문을 가져오되, 모든 원문을 다 넣지 않고 점수가 높은 evidence block 중심으로 context를 만듭니다.
- LLM prompt에는 "선별 근거와 필드 후보에 있는 숫자/날짜/서류명만 사용"하라는 strict grounding 규칙을 추가했습니다.

비교 방식은 간단합니다. 기존 최고 후보였던 `E_fields_v2`와 새 `G_context_builder_v2`를 같은 대표 오답 30개에 돌려서 hallucination이 줄고 정답 근거 선택이 좋아지는지 확인하면 됩니다.


In [16]:
# 7. Run generation experiments on selected subset
import csv

RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
LOCAL_EXPERIMENT_DIR = PROJECT_DIR / 'outputs/generation_improvement_experiments' / RUN_TAG
LOCAL_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

_generator_cache = {}
def get_generator(model_name):
    if model_name not in _generator_cache:
        _generator_cache[model_name] = HuggingFaceGenerator(
            model_name=model_name,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
        )
    return _generator_cache[model_name]

def csv_cell(value):
    return json.dumps(value, ensure_ascii=False)

def write_review_csv(path, payloads):
    fieldnames = [
        'variant', 'id', 'type', 'difficulty', 'question_type', 'question',
        'generated_answer', 'ground_truth_answer', 'ground_truth_docs',
        'retrieved_docs', 'source_store_ids', 'field_candidates', 'guardrail_warnings',
        'human_correctness', 'evidence_grounded', 'failure_type', 'review_memo'
    ]
    with path.open('w', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for payload in payloads:
            writer.writerow({
                'variant': payload['variant'],
                'id': payload['id'],
                'type': payload['type'],
                'difficulty': payload['difficulty'],
                'question_type': payload['question_type'],
                'question': payload['question'],
                'generated_answer': payload['answer'],
                'ground_truth_answer': payload['ground_truth_answer'],
                'ground_truth_docs': csv_cell(payload['ground_truth_docs']),
                'retrieved_docs': csv_cell(payload['retrieved_docs']),
                'source_store_ids': csv_cell(payload['source_store_ids']),
                'field_candidates': csv_cell(payload['field_candidates']),
                'guardrail_warnings': csv_cell(payload['guardrail'].get('warnings', [])),
                'human_correctness': '',
                'evidence_grounded': '',
                'failure_type': '',
                'review_memo': '',
            })

generator = get_generator(MODEL_NAME)
all_payloads = []

for variant_name in VARIANTS_TO_RUN:
    variant = EXPERIMENT_VARIANTS[variant_name]
    jsonl_path = LOCAL_EXPERIMENT_DIR / f'{variant_name}.jsonl'
    variant_payloads = []
    with jsonl_path.open('w', encoding='utf-8') as file:
        for index, row in enumerate(rows, 1):
            started = time.perf_counter()
            bundle = render_prompt(row, variant)
            answer = generator.generate_prompt(bundle['prompt'])
            answer = dedupe_repeated_lines(answer)
            checks = guardrail(answer, bundle, variant)
            payload = {
                'variant': variant_name,
                'variant_description': variant['description'],
                'id': row['id'],
                'type': row['type'],
                'difficulty': row['difficulty'],
                'question': row['question'],
                'question_type': bundle['question_type'],
                'answer': answer,
                'ground_truth_answer': row['ground_truth_answer'],
                'ground_truth_docs': row['ground_truth_docs'],
                'retrieved_docs': [record.get('filename') for record in bundle['context_records']],
                'source_store_ids': [record.get('source_store_id') for record in bundle['source_records']],
                'field_candidates': bundle['field_candidates'],
                'guardrail': checks,
                'latency_ms': int((time.perf_counter() - started) * 1000),
                'model': MODEL_NAME,
                'prompt': bundle['prompt'],
            }
            file.write(json.dumps(payload, ensure_ascii=False) + '\n')
            variant_payloads.append(payload)
            all_payloads.append(payload)
            print(f'[{variant_name}] {index}/{len(rows)} {row["id"]} warnings={checks.get("warnings")}')
    write_review_csv(LOCAL_EXPERIMENT_DIR / f'{variant_name}_manual_review.csv', variant_payloads)

write_review_csv(LOCAL_EXPERIMENT_DIR / 'all_variants_manual_review.csv', all_payloads)
print('Local experiment dir:', LOCAL_EXPERIMENT_DIR)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[E_fields_v2] 1/30 Q006 warnings=[]
[E_fields_v2] 2/30 Q017 warnings=['unknown_answer_despite_field_candidates']
[E_fields_v2] 3/30 Q020 warnings=[]
[E_fields_v2] 4/30 Q031 warnings=['missing_source_citation']
[E_fields_v2] 5/30 Q123 warnings=[]
[E_fields_v2] 6/30 Q137 warnings=[]
[E_fields_v2] 7/30 Q001 warnings=['unknown_answer_despite_field_candidates']
[E_fields_v2] 8/30 Q013 warnings=['missing_source_citation']
[E_fields_v2] 9/30 Q076 warnings=[]
[E_fields_v2] 10/30 Q104 warnings=['unknown_answer_despite_field_candidates']
[E_fields_v2] 11/30 Q088 warnings=['answer_has_number_or_date_not_in_context']
[E_fields_v2] 12/30 Q007 warnings=['answer_has_number_or_date_not_in_context']
[E_fields_v2] 13/30 Q008 warnings=['answer_has_number_or_date_not_in_context']
[E_fields_v2] 14/30 Q016 warnings=[]
[E_fields_v2] 15/30 Q026 warnings=['answer_has_number_or_date_not_in_context']
[E_fields_v2] 16/30 Q028 warnings=[]
[E_fields_v2] 17/30 Q003 warnings=['missing_source_citation', 'unknown_answe

In [17]:
# 8. Save experiment outputs to Drive
DRIVE_EXPERIMENT_DIR = DRIVE_EXPERIMENT_ROOT / RUN_TAG
DRIVE_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
for path in sorted(LOCAL_EXPERIMENT_DIR.glob('*')):
    if path.suffix.lower() in {'.jsonl', '.csv', '.md'}:
        shutil.copy2(path, DRIVE_EXPERIMENT_DIR / path.name)
print('Drive experiment dir:', DRIVE_EXPERIMENT_DIR)

Drive experiment dir: /content/drive/MyDrive/chatbot_colab_outputs/generation_improvement_experiments/20260526_083037


## 5. Human Evaluation 기준

`*_manual_review.csv`에는 아래 컬럼을 사람이 채웁니다.

`human_correctness` 권장 값:
- `correct`: 정답과 핵심 내용이 맞음
- `partial`: 일부만 맞거나 세부 조건/목록이 누락됨
- `wrong`: 핵심 답이 틀림
- `unanswerable_correct`: 확인 불가 질문에서 확인 불가로 잘 답함
- `unclear`: 정답/근거 자체가 애매함

`evidence_grounded` 권장 값:
- `yes`: 답변이 제시된 context와 근거 문장에 의해 지지됨
- `partial`: 답은 맞지만 근거가 약하거나 일부만 지지됨
- `no`: context에 없는 내용을 답함
- `missing_citation`: 답은 있어 보이나 문서명/근거 문장이 누락됨

`failure_type` 권장 값:
- `retrieval_miss`
- `context_too_short`
- `source_store_needed`
- `table_detail_missing`
- `wrong_field_choice`
- `missed_candidate_value`
- `generation_hallucination`
- `date_or_budget_format_error`
- `submission_list_incomplete`
- `unanswerable_mishandled`
- `citation_missing_or_wrong`
- `ground_truth_or_eval_issue`

처음에는 각 variant별 20개만 보고, `question_type x failure_type` 분포를 비교합니다.

In [18]:
# 6. Judge LLM 자동 평가 설계 및 optional 실행
JUDGE_SCHEMA = {
    'correctness': 'correct|partial|wrong|unanswerable_correct|unclear',
    'evidence_grounded': 'yes|partial|no|missing_citation',
    'failure_type': 'one of the agreed failure_type labels',
    'reason': 'short Korean explanation',
}

def build_judge_prompt(payload):
    return f"""
너는 RFP QA 결과를 평가하는 Judge다. 아래 기준으로만 평가하고 JSON 하나만 출력한다.

정답성(correctness): correct, partial, wrong, unanswerable_correct, unclear
근거성(evidence_grounded): yes, partial, no, missing_citation
failure_type 후보: retrieval_miss, context_too_short, source_store_needed, table_detail_missing, wrong_field_choice, missed_candidate_value, generation_hallucination, date_or_budget_format_error, submission_list_incomplete, unanswerable_mishandled, citation_missing_or_wrong, ground_truth_or_eval_issue

[Question]
{payload.get('question')}

[Generated Answer]
{payload.get('answer')}

[Ground Truth Answer]
{payload.get('ground_truth_answer')}

[Retrieved Docs]
{payload.get('retrieved_docs')}

[Field Candidates]
{json.dumps(payload.get('field_candidates') or {}, ensure_ascii=False)[:3000]}

[Guardrail]
{json.dumps(payload.get('guardrail') or {}, ensure_ascii=False)}

출력 JSON schema:
{json.dumps(JUDGE_SCHEMA, ensure_ascii=False)}
""".strip()

def try_parse_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
    return {'parse_error': True, 'raw': text}

if RUN_JUDGE:
    judge = get_generator(JUDGE_MODEL_NAME)
    judged = []
    for payload in all_payloads:
        judge_answer = judge.generate_prompt(build_judge_prompt(payload))
        payload = dict(payload)
        payload['judge'] = try_parse_json(judge_answer)
        judged.append(payload)
        print(payload['variant'], payload['id'], payload['judge'])
    judge_path = LOCAL_EXPERIMENT_DIR / 'judge_results.jsonl'
    with judge_path.open('w', encoding='utf-8') as file:
        for payload in judged:
            file.write(json.dumps(payload, ensure_ascii=False) + '\n')
    shutil.copy2(judge_path, DRIVE_EXPERIMENT_DIR / judge_path.name)
else:
    print('RUN_JUDGE=False. Judge prompt/schema만 준비했습니다.')

RUN_JUDGE=False. Judge prompt/schema만 준비했습니다.


## 7. 모델 변경 또는 튜닝 실험 순서

모델 변경은 context/source_store/fields/guardrail 실험 후에 진행합니다. 같은 subset, 같은 variant로 모델만 바꿔야 원인 분석이 됩니다.

추천 순서:
1. `Qwen/Qwen2.5-3B-Instruct` 현재 baseline 체급
2. `Qwen/Qwen3-4B-Instruct-2507` 4B 체급 업그레이드
3. `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct` 한국어 특화 2.4B 비교
4. `meta-llama/Llama-3.2-3B-Instruct` Llama 3B 비교군
5. GPU 여유가 있으면 `Qwen/Qwen2.5-7B-Instruct` 또는 `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct`

우선은 `A_baseline_like_context1000` vs `C_source_store_basic` vs `E_fields_v2`를 같은 20개 문제에 돌리고, 사람이 20~60행만 비교 채점해서 source_store와 field 후보 주입의 효과를 먼저 확인합니다.